# Infinity LLM — independent Google Colab GPU continuation

This notebook trains a **separate copy**, in parallel with the existing GitHub job. It does not cancel, modify, publish over, or merge into that job, and it does not automatically deploy a model. It starts from the saved `checkpoint-16-1.pt` (step 536,897; 68,722,816 sampled positions) and the same checksum-verified `books100m-v1` dataset. The GitHub run may have advanced since that snapshot.

**How to use**: In Colab choose **Runtime → Change runtime type → GPU**, then run the cells in order. Sign in and approve the Google Drive mount when prompted. The notebook needs at least 5 GB of local disk (more after interrupted segments) plus space in Drive for each 0.9 GB checkpoint; Drive storage may fill. Only use one notebook session per backup folder. No GitHub token or password is needed.

The training cell runs four 15-minute segments by default, backing up after each successful segment. To continue, rerun that cell (or open this notebook in a new GPU runtime and run all cells). A sudden disconnection or quota expiry can lose progress since the last completed Drive backup; backups can fail if Drive is full. They are not deleted automatically. Free Colab GPUs and session lengths are not guaranteed. Keep your notebook tab open and check that backups finish and appear in Drive; mounted Drive writes can fail or be delayed. Backups are pickle files: **only resume files created by this notebook in your own trusted Drive folder**.

It restores the optimizer, cycle and CPU random states. GPU arithmetic and precision differ from the CPU run, and CUDA random/FP16 scaler state are not preserved by this trainer, so the result is **not bitwise identical**. More training does not guarantee better web-informed answers.


In [ ]:
import sys, os, subprocess
from pathlib import Path
import torch, numpy
if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. In Colab: Runtime → Change runtime type → GPU, then reconnect and rerun.')
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.get_device_name(0), '| GPU BF16:', torch.cuda.is_bf16_supported())
from google.colab import drive
drive.mount('/content/drive')
BASE = Path('/content/infini-colab-parallel')
REPO = BASE / 'code'
DATA = BASE / 'datasets' / 'books100m'
RUN = BASE / 'run'
BACKUP = Path('/content/drive/MyDrive/Infini-Colab-Parallel-books100m-43698262111')
for directory in (BASE, RUN, BACKUP): directory.mkdir(parents=True, exist_ok=True)
print('Backups:', BACKUP)


## Download and verify the exact training code, dataset and initial checkpoint

The checkpoint is a public file of about 900 MB. Download time depends on the Colab connection. Checksums are checked before use; nothing is uploaded to GitHub.


In [ ]:
import hashlib, urllib.request, tarfile
TRAIN_COMMIT = '1c33e90d654ad48b0c96102474943dbd1af1f0ae'
CKPT_SHA = 'f29f0de64a6ab1b20f260f05e8d9c141e1ae1c9786b789568ac13ce70e3a3426'
DATA_SHA = 'bc73cd70757479c21da2bbbcfcd0d36b3c374ed126a35668cb0c295c274d30e3'
RELEASE = 'https://github.com/Yuvii123-bot/InFiNi/releases/download/'

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            chunk = f.read(8 * 1024 * 1024)
            if not chunk: break
            h.update(chunk)
    return h.hexdigest()

def download_checked(url, destination, expected):
    if destination.exists() and sha256_file(destination) == expected:
        print('Verified cached', destination.name); return
    temporary = destination.with_name(destination.name + '.download')
    try:
        request = urllib.request.Request(url, headers={'User-Agent': 'infini-colab'})
        h = hashlib.sha256()
        with urllib.request.urlopen(request, timeout=120) as src, open(temporary, 'wb') as dst:
            while True:
                chunk = src.read(8 * 1024 * 1024)
                if not chunk: break
                dst.write(chunk); h.update(chunk)
        if h.hexdigest() != expected: raise ValueError('Checksum mismatch: ' + destination.name)
        os.replace(temporary, destination)
    finally:
        if temporary.exists(): temporary.unlink()
    print('Downloaded and verified', destination.name)

if not REPO.exists():
    subprocess.run(['git','clone','--no-checkout','https://github.com/Yuvii123-bot/InFiNi.git',str(REPO)], check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','--detach',TRAIN_COMMIT], check=True)
head = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip()
if head != TRAIN_COMMIT: raise RuntimeError('Unexpected training code commit: ' + head)
# Do not run modified training code under the original commit identity.
if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain','--','llm'], text=True).strip():
    raise RuntimeError('Local training code was modified; use a clean runtime.')
checkpoint = BASE / 'checkpoint-16-1.pt'
archive = BASE / 'dataset-books100m.tar.gz'
download_checked(RELEASE + 'infinity-qa75m-44205300285/checkpoint-16-1.pt', checkpoint, CKPT_SHA)
download_checked(RELEASE + 'infinity-books100m-data-43697791940/dataset-books100m.tar.gz', archive, DATA_SHA)
# Overwrite the local working dataset with the verified fixed archive on each setup.
(BASE / 'datasets').mkdir(parents=True, exist_ok=True)
subprocess.run(['tar', '-xzf', str(archive), '-C', str(BASE / 'datasets')], check=True)
if not (DATA / 'dataset.json').is_file(): raise RuntimeError('Dataset archive did not contain the expected manifest')
print('Training code:', head, '| Fixed dataset:', DATA)


## Resume the separate copy and back it up to your Drive

The newest checksum-valid notebook backup in the folder is resumed; otherwise the verified GitHub snapshot is copied locally. Every run uses the same dataset, original cycle, optimizer and learning schedule. The default 4 × 15-minute segments keep validation/backup overhead modest. A backup is a separate full checkpoint, not a merge with GitHub. Once Drive space is limited, move or remove **older** backups manually, retaining at least the newest verified one. For another session, reconnect a GPU and run the notebook from the top.


In [ ]:
import json, re, shutil, time, math
# Do not run two sessions against the same backup folder.
SEGMENTS = 4
SECONDS_PER_SEGMENT = 900
assert 1 <= SEGMENTS <= 24 and 60 <= SECONDS_PER_SEGMENT <= 3600
pattern = re.compile(r'^checkpoint-step-(\d+)-([0-9a-f]{64})\.pt$')

def checkpoint_info(path):
    # Checkpoint loading is intentionally restricted to the pinned GitHub file or your trusted own backups.
    state = torch.load(path, map_location='cpu', weights_only=False, mmap=True)
    cycle = state.get('cycle') or {}
    if cycle.get('id') != 'books100m-v1' or cycle.get('start_step') != 500 or cycle.get('steps') != 781250:
        raise RuntimeError('Checkpoint belongs to a different training cycle')
    if not state.get('optimizer') or not state.get('np_rng') or state.get('torch_rng') is None:
        raise RuntimeError('Checkpoint is missing the optimizer or CPU random state')
    step = int(state['step'])
    if not 536897 <= step <= 781750: raise RuntimeError('Unexpected checkpoint step: ' + str(step))
    return step, int(state['tokens_seen'])

def backup_checkpoint(source):
    step, tokens = checkpoint_info(source)
    digest = sha256_file(source)
    destination = BACKUP / f'checkpoint-step-{step:09d}-{digest}.pt'
    if destination.exists():
        if sha256_file(destination) != digest: raise RuntimeError('Existing backup has a checksum mismatch')
    else:
        temporary = BACKUP / (destination.name + '.partial')
        try:
            shutil.copyfile(source, temporary)
            if sha256_file(temporary) != digest: raise RuntimeError('Drive backup verification failed')
            os.replace(temporary, destination)
        finally:
            if temporary.exists(): temporary.unlink()
    if sha256_file(destination) != digest: raise RuntimeError('Published Drive backup verification failed; previous backups are unchanged')
    print(f'Verified Drive backup: step {step:,}, total sampled positions {tokens:,}\n{destination}', flush=True)
    return step

local_checkpoint = RUN / 'ckpt.pt'
pending = RUN / 'segment-in-progress'
if pending.exists():
    # The last segment or Drive copy did not finish. Never silently promote its local checkpoint.
    if local_checkpoint.exists():
        quarantine = RUN / f'ckpt-unverified-{time.time_ns()}.pt'
        os.replace(local_checkpoint, quarantine)
        print('Unverified interrupted segment retained locally:', quarantine)
    pending.unlink()
    print('Resuming from the most recent verified Drive backup (or the original snapshot).')
backups = sorted(((int(m.group(1)), p, m.group(2)) for p in BACKUP.iterdir() if (m := pattern.fullmatch(p.name))), reverse=True)
# On rerun, choose the newest verified local or Drive state, never silently replace newer progress.
local_step = checkpoint_info(local_checkpoint)[0] if local_checkpoint.exists() else -1
if len(backups) > 1 and backups[0][0] == backups[1][0] and backups[0][2] != backups[1][2]:
    raise RuntimeError('Multiple different backups have the same newest step; choose one manually before continuing.')
if backups and backups[0][0] == local_step:
    if sha256_file(backups[0][1]) != backups[0][2]: raise RuntimeError('Newest Drive backup is corrupted; inspect it before continuing.')
    if sha256_file(local_checkpoint) != backups[0][2]:
        raise RuntimeError('Local and Drive checkpoints differ at the same step; choose one manually before continuing.')
if backups and backups[0][0] > local_step:
    step, restore, expected = backups[0]
    if sha256_file(restore) != expected: raise RuntimeError('Newest Drive backup is corrupted; inspect it before continuing.')
    if checkpoint_info(restore)[0] != step: raise RuntimeError('Backup filename and checkpoint disagree.')
    temporary = RUN / 'ckpt.restore'
    shutil.copyfile(restore, temporary)
    if sha256_file(temporary) != expected: raise RuntimeError('Restored copy checksum mismatch')
    os.replace(temporary, local_checkpoint)
elif local_step < 0:
    if sha256_file(checkpoint) != CKPT_SHA or checkpoint_info(checkpoint)[0] != 536897: raise RuntimeError('Initial checkpoint verification failed')
    shutil.copyfile(checkpoint, local_checkpoint)
current_step = checkpoint_info(local_checkpoint)[0]
if current_step > 536897 and (not backups or current_step > backups[0][0]):
    print('Saving locally recovered progress to Drive before continuing.')
    backup_checkpoint(local_checkpoint)
print('Resuming separate Colab copy at step', current_step)

settings = {'batch_size': 1, 'sequence_length': 128, 'threads': 2, 'device': 'cuda', 'precision': 'auto',
            'lr': 0.00003, 'min_lr': 0.000003, 'warmup_steps': 1000, 'eval_interval': 5000,
            'eval_windows': 16, 'log_interval': 250, 'save_interval': 1000, 'nice': 0}
for segment in range(SEGMENTS):
    before, _ = checkpoint_info(local_checkpoint)
    if before >= 781750:
        print('The original 100M-position cycle is complete for this Colab copy.'); break
    cmd = [sys.executable, '-m', 'llm.train', '--data', str(DATA), '--run', str(RUN), '--steps', '781250',
           '--max-seconds', str(SECONDS_PER_SEGMENT), '--cycle', 'books100m-v1']
    for key, value in settings.items(): cmd.extend(['--set', f'{key}={value}'])
    print(f'Segment {segment+1}/{SEGMENTS}: training on GPU from step {before:,}...', flush=True)
    pending.write_text('A segment or its Drive backup is not yet verified.\n')
    result = subprocess.run(cmd, cwd=REPO)
    if result.returncode != 0: raise RuntimeError('Training failed; prior Drive backups are unchanged. Exit ' + str(result.returncode))
    status = json.loads((RUN / 'status.json').read_text())
    if status.get('device') != 'cuda' or not status.get('optimizer_state_restored') or not str(status.get('source', '')).startswith('resumed '):
        raise RuntimeError('Trainer did not confirm GPU and optimizer restore; refusing to save as a successful continuation.')
    if status.get('val_loss') is None or not math.isfinite(status['val_loss']):
        raise RuntimeError('Validation loss is not finite; prior Drive backups are unchanged.')
    after = checkpoint_info(local_checkpoint)[0]
    if int(status.get('step', -1)) != after: raise RuntimeError('Status and checkpoint steps disagree.')
    if after <= before: raise RuntimeError('No training progress; prior Drive backups are unchanged.')
    backup_checkpoint(local_checkpoint)
    pending.unlink()
    if after >= 781750:
        print('The original 100M-position cycle is complete for this Colab copy.'); break
print('Done. Run this cell again to continue. The GitHub job was not touched or replaced.')
